## Testing the new parser code to load models with different options

In [1]:
from pycbc.workflow import WorkflowConfigParser
from pycbc.inference.models.gaussian_noise import GaussianNoise
from pycbc.inference.models.marginalized_gaussian_noise import MarginalizedHMPhase
from pycbc.inference.models import read_from_config
import h5py
import numpy as np
import matplotlib.pyplot as plt

/Users/vikasjadhav/anaconda3/envs/hmnumerical/lib/python3.11/site-packages/pycbc/types/array.py:36: UserWarning: Wswiglal-redir-stdio:

SWIGLAL standard output/error redirection is enabled in IPython.
This may lead to performance penalties. To disable locally, use:

with lal.no_swig_redirect_standard_output_error():
    ...

To disable globally, use:

lal.swig_redirect_standard_output_error(False)

Note however that this will likely lead to error messages from
LAL functions being either misdirected or lost when called from
Jupyter notebooks.

To suppress this warning, use:

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal

  import lal as _lal
PyCBC.libutils: pkg-config call failed, setting NO_PKGCONFIG=1


In [2]:

def create_parser(
        injection_file,variable_params_file,data_file,model_file,
        mode_array,**kwargs):
    """ 
    Create a Workflowconfigparser that updates static_params by reading it from 
    the injection file. 
    Inputs : 
    "injection file" : an hdf file containing only one injection
    "variable_params_file" : a config file containing section of variable_params
    and must include the priors of each parameter.
    "data_file" : a config file containing common data settings.The trigger-time and 
    injection-file options will be updated using the provided injection.
    "model_file" : config file with the model settings.
    "mode_array" : mode array to use in the signal model.
    "kwargs" : additional options for the [model] section.
    Output : 
    "cp" : a workflow config parser
    """
    cp = WorkflowConfigParser([data_file,variable_params_file,model_file])
    
    
    ## Read the injection file to gather all the params
    inj = h5py.File(injection_file,'r')
    all_params = {}
    for p in inj.keys():
        all_params[p] = inj[p][0]
    for sp in inj.attrs['static_args']:
        all_params[sp] = inj.attrs[sp]
    static_params = all_params.copy()
    ## Remove the params included in the variable params
    for vp in cp['variable_params'].keys():
        _ = static_params.pop(vp)
    
    ## Add the mode array to be used in the model
    static_params['mode_array'] = mode_array
    ## Create and add options to static_params section
    cp.add_section('static_params')
    for sp in static_params:
        cp.add_options_to_section('static_params',
                                  [(sp,str(static_params[sp]))])
    
    ## Update options to data section
    cp.add_options_to_section('data',[('trigger-time',str(static_params['tc'])),
                                      ('injection-file',injection_file)])
    for key, value in kwargs.items():
        cp.add_options_to_section('model', [(key, str(value))])
    return cp




In [3]:
injection_file = '../injections/general_pop/injection_0.hdf'
test_injection = h5py.File(injection_file,'r')

for p in test_injection.keys():
    print(p, np.array(test_injection[p]))
for p in test_injection.attrs['static_args']:
    print(p,test_injection.attrs[p])

coa_phase [2.74944154]
dec [0.90035254]
distance [844.91105243]
inclination [1.66068369]
mass1 [29.69660768]
mass2 [37.18352149]
polarization [2.66190161]
ra [4.0582724]
approximant IMRPhenomXPHM
f_ref 20.0
f_lower 20.0
tc 1420871141.223


In [4]:
cp = create_parser(injection_file=injection_file,
                        variable_params_file='../config_files/var_par.ini',
                        data_file='../config_files/data_config/data_noise.ini',
                        model_file='../config_files/model_config/hm_model.ini',
                        mode_array='22 33')

cp_dp = create_parser(injection_file=injection_file,
                        variable_params_file='../config_files/var_par.ini',
                        data_file='../config_files/data_config/data_noise.ini',
                        model_file='../config_files/model_config/hm_model.ini',
                        mode_array='22 33',
                        dominant_mode_peak=True)


In [5]:
hm = read_from_config(cp)
hm_dp = read_from_config(cp_dp)

In [6]:
hm.update(coa_phase = 0)
hm_dp.update(coa_phase=0)

In [7]:
print(hm.loglr)
print(hm_dp.loglr)

using analytic approximation
0.00040883407928049564
39.59528570888898
using analytic approximation
0.0002779171336442232
39.5903361336527


In [8]:
print(hm.shm)

{2: np.complex128(52.419074313523296+54.81020809412182j), 3: np.complex128(-0.41004413077778595+0.6068962013223816j)}


In [9]:
print(np.sort(hm.peaks))
print(np.sort(hm_dp.peaks))

[1.16254339 2.74361196 4.31279384 5.87361643]
[1.16694946 2.73774579 4.30854212 5.87933844]


In [12]:
def log_rel_err(log_approx, log_true):
    """
    Calculates absolute relative error between approx and true values
    |approx-true|/true
    """
    log_approx = np.asarray(log_approx)
    log_true = np.asarray(log_true)
    delta = log_approx - log_true
    print(delta)
    result = np.where(delta > 0,
                      np.log10(np.exp(delta) - 1),
                      np.log10(1 - np.exp(delta)))
    return result